In [ ]:
# 16_honest_did.ipynb
# Rambachan-Roth honest difference-in-differences sensitivity and a formal joint test
# that the Callaway-Sant'Anna pre-period coefficients are zero, for the overall CHE40
# effect and for each baseline-PHI group (the buffer). Writes t18_honest_did.csv and
# fig18 (grayscale). Requires moderndid[didhonest] (cvxpy, highspy).
# CS pre-trend joint Wald + Rambachan-Roth honest-DiD (relative magnitudes) breakdown,
# for the overall CHE40 effect and for each baseline-PHI group (the buffer). One estimator.
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, moderndid as md
from scipy.stats import chi2
PROC, TAB = "../data/processed", "../results/tables"
p=pd.read_parquet(f"{PROC}/khp_panel.parquet"); p["che40f"]=p["che40"].astype(float)
dd=p[["PIDWON","year","cohort","che40f","base_phi"]].dropna(subset=["che40f"])

def fit(df):
    mp=md.att_gt(data=df,yname="che40f",tname="year",idname="PIDWON",gname="cohort",
                 control_group="notyettreated",est_method="dr",base_period="universal",
                 panel=True,allow_unbalanced_panel=True,boot=False)
    return md.aggte(mp,type="dynamic",na_rm=True)

def pretrend(dyn):
    et=np.asarray(dyn.event_times); se=np.asarray(dyn.se_by_event); att=np.asarray(dyn.att_by_event)
    IF=np.asarray(dyn.influence_func); n=IF.shape[0]; V=IF.T@IF/n**2   # covariance of the event-study estimates
    pre=(et<0)&(se>0)                      # exclude the normalized reference period (se=0)
    idx=np.where(pre)[0]; b=att[idx]; Vp=V[np.ix_(idx,idx)]
    stat=float(b@np.linalg.pinv(Vp)@b); k=len(idx)
    return stat,k,float(1-chi2.cdf(stat,k))

def rr(dyn, mbar=(0.5,1.0,1.5,2.0,3.0)):
    hd=md.honest_did(dyn,event_time=0,sensitivity_type="relative_magnitude",m_bar_vec=list(mbar))
    rc=hd.robust_ci.to_pandas() if hasattr(hd.robust_ci,"to_pandas") else pd.DataFrame(hd.robust_ci)
    oc=hd.original_ci
    out=[]; bd=None
    for _,r in rc.iterrows():
        lb,ub,mb=float(r["lb"]),float(r["ub"]),float(r["Mbar"])
        inc = not (lb>0 or ub<0)
        out.append((mb,lb,ub,inc))
        if bd is None and inc: bd=mb
    return float(oc.lb),float(oc.ub),out,bd

rows=[]
for key,lab in [(None,"Full sample"),(0,"No PHI"),(1,"Has PHI")]:
    df = dd if key is None else dd[dd["base_phi"]==key]
    dyn=fit(df)
    st,k,pv=pretrend(dyn)
    olb,oub,rrtab,bd=rr(dyn)
    k0=float(np.asarray(dyn.att_by_event)[np.asarray(dyn.event_times)==0][0])
    print(f"\n=== {lab} === CHE40 effect at k=0 = {k0:+.4f}")
    print(f"  CS pre-trend joint Wald: chi2({k})={st:.3f}, p={pv:.4f}")
    print(f"  original 95% CI: [{olb:+.4f}, {oub:+.4f}]")
    for mb,lb,ub,inc in rrtab:
        print(f"  RR Mbar={mb:>3}: [{lb:+.4f}, {ub:+.4f}] {'INCLUDES 0' if inc else 'excludes 0'}")
    print(f"  breakdown Mbar = {bd if bd is not None else '>3 (robust across all tested)'}")
    for mb,lb,ub,inc in rrtab: rows.append({"group":lab,"mbar":mb,"lb":lb,"ub":ub,"includes_0":inc})
    rows.append({"group":lab,"mbar":"pretrend_p","lb":pv,"ub":st,"includes_0":k})
pd.DataFrame(rows).to_csv(f"{TAB}/t18_honest_did.csv",index=False)
print("\nsaved t18_honest_did.csv")


# ---- Figure 18: honest-DiD robust bands ----
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
FIG,TAB="../results/figures","../results/tables"
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"font.size":11,"axes.edgecolor":"0.3","savefig.dpi":600,"figure.dpi":110,"grid.color":"0.9"})
t=pd.read_csv(f"{TAB}/t18_honest_did.csv")
t=t[t["mbar"].apply(lambda v: str(v).replace('.','',1).isdigit())].copy(); t["mbar"]=t["mbar"].astype(float)
fig,ax=plt.subplots(figsize=(7.2,4.4))
styles={"No PHI":("0.12","o-","white"),"Has PHI":("0.55","s--","0.55")}
for grp,(shade,fmt,mf) in styles.items():
    g=t[t["group"]==grp].sort_values("mbar")
    mid=(g["lb"]+g["ub"])/2
    ax.fill_between(g["mbar"],g["lb"],g["ub"],color=shade,alpha=0.15)
    ax.errorbar(g["mbar"],mid,yerr=[(mid-g["lb"]),(g["ub"]-mid)],fmt=fmt,color=shade,ms=6,lw=1.6,
                capsize=3,ecolor=shade,markerfacecolor=mf,markeredgewidth=1.3,label=grp)
ax.axhline(0,color="0.5",lw=0.9)
ax.set_xlabel(r"Relative-magnitude bound $\bar{M}$ (post violation / max pre-period violation)")
ax.set_ylabel("Honest-DiD robust 95% CI, effect on P(CHE 40%) at k=0")
ax.legend(frameon=False,title="Baseline insurance")
for ext in ("png","pdf"): fig.savefig(f"{FIG}/fig18_honest_did.{ext}",dpi=600,bbox_inches="tight")
plt.close(fig); print("fig18 written")
